In [1]:
import pandas as pd
import altair as alt

alt.theme.enable("latimes")

df = pd.read_csv("Benchmark.Benchmark-report.csv", delimiter=",")


def clean_data(df):
    df["Median"] = df["Median"].str.replace(" ms", "", case=False, regex=False)
    df["Median"] = df["Median"].str.replace(",", "", case=False, regex=False)
    df = df.astype({"Median": "float64"})
    # ensure the resource count is numeric for plotting
    df["ResourcesCount"] = pd.to_numeric(df["ResourcesCount"], errors="coerce")
    return df


df_clean = clean_data(df.copy())

# rename methods for readability
method_map = {
    "AllResources_Default": "Postgres",
    "AllResources_Hypertable": "TimescaleDB",
}
df_clean["Method"] = df_clean["Method"].map(method_map)

defaults = df_clean[df_clean["Method"] == "Postgres"].set_index("ResourcesCount")[
    "Median"
]


def median_ratio(row):
    if row["Method"] == "TimescaleDB":
        base = defaults.loc[row["ResourcesCount"]]
        return (row["Median"] / base - 1) * 100
    else:
        return pd.NA


df_clean["Ratio"] = df_clean.apply(median_ratio, axis=1)

# Main chart: Execution time
main_plot = (
    alt.Chart(df_clean)
    .mark_line()
    .encode(
        y=alt.Y("Median", sort="x").title("Median execution time, ms"),
        x=alt.X("ResourcesCount")
        .scale(type="log")
        .title("Number of resources (log scale)"),
        color="Method",
    )
    .properties(
        title="TimescaleDB vs PostgreSQL: Execution time comparison",
        height=300,
        width=800,
    )
)

# Ratio chart – only hypertable line
ratio_plot = (
    alt.Chart(df_clean[df_clean["Method"] == "TimescaleDB"])
    .mark_line(strokeDash=[4, 2])
    .encode(
        y=alt.Y("Ratio", title="Ratio (%)"),
        x=alt.X("ResourcesCount")
        .scale(type="log")
        .title("Number of resources (log scale)"),
    )
    .properties(
        height=150,
        width=800,
    )
)

# Combine vertically
final_plot = alt.vconcat(main_plot, ratio_plot)
final_plot.show()
final_plot.save("results.png", scale_factor=2)

alt.VConcatChart(...)